<a href="https://colab.research.google.com/github/rahu2004/SQL/blob/main/Student_Management_System_using_MySQL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
!apt-get update -qq
!apt-get install mysql-server -y

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
mysql-server is already the newest version (8.0.45-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 17 not upgraded.


In [6]:
!service mysql start

 * Starting MySQL database server mysqld
su: warning: cannot change directory to /nonexistent: No such file or directory
   ...done.


In [7]:
!service mysql status

 * /usr/bin/mysqladmin  Ver 8.0.45-0ubuntu0.22.04.1 for Linux on x86_64 ((Ubuntu))
Copyright (c) 2000, 2026, Oracle and/or its affiliates.

Oracle is a registered trademark of Oracle Corporation and/or its
affiliates. Other names may be trademarks of their respective
owners.

Server version		8.0.45-0ubuntu0.22.04.1
Protocol version	10
Connection		Localhost via UNIX socket
UNIX socket		/var/run/mysqld/mysqld.sock
Uptime:			9 sec

Threads: 2  Questions: 8  Slow queries: 0  Opens: 119  Flush tables: 3  Open tables: 38  Queries per second avg: 0.888


In [8]:
!mkdir -p /var/run/mysqld
!chown mysql:mysql /var/run/mysqld

In [9]:
!service mysql restart

 * Stopping MySQL database server mysqld
   ...done.
 * Starting MySQL database server mysqld
su: warning: cannot change directory to /nonexistent: No such file or directory
   ...done.


In [10]:
!mysql -u root -e "SELECT VERSION();"

+-------------------------+
| VERSION()               |
+-------------------------+
| 8.0.45-0ubuntu0.22.04.1 |
+-------------------------+


In [11]:
!mysql -u root -e "ALTER USER 'root'@'localhost' IDENTIFIED WITH mysql_native_password BY 'root'; FLUSH PRIVILEGES;"

connecting mysql using python


In [15]:
import mysql.connector

conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="root"
)
cursor = conn.cursor()

print("Connected Successfully")

Connected Successfully


create database

In [16]:
cursor.execute(
    "CREATE DATABASE student_management"
)

print("Database Created")

Database Created


use database

In [17]:
cursor.execute(
    "USE student_management"
)

Full SQL Script

In [20]:
sql_script = """
-- CREATE STUDENTS TABLE

CREATE TABLE Students (
    student_id INT PRIMARY KEY,
    student_name VARCHAR(100),
    department VARCHAR(50),
    age INT,
    city VARCHAR(50)
);

-- CREATE COURSES TABLE

CREATE TABLE Courses (
    course_id INT PRIMARY KEY,
    course_name VARCHAR(100),
    faculty_name VARCHAR(100)
);

-- CREATE ENROLLMENTS TABLE

CREATE TABLE Enrollments (
    enrollment_id INT PRIMARY KEY,
    student_id INT,
    course_id INT,
    marks INT,

    FOREIGN KEY(student_id)
    REFERENCES Students(student_id),

    FOREIGN KEY(course_id)
    REFERENCES Courses(course_id)
);

-- INSERT STUDENT DATA

INSERT INTO Students VALUES
(1, 'Rahul', 'AI', 21, 'Chennai'),
(2, 'Kiran', 'Data Science', 22, 'Hyderabad'),
(3, 'Anjali', 'Cyber Security', 20, 'Bangalore'),
(4, 'Rohit', 'AI', 23, 'Mumbai');

-- INSERT COURSE DATA

INSERT INTO Courses VALUES
(101, 'Machine Learning', 'Dr. Kumar'),
(102, 'Data Analytics', 'Dr. Priya'),
(103, 'Cloud Computing', 'Dr. Sharma');


-- INSERT ENROLLMENT DATA

INSERT INTO Enrollments VALUES
(1, 1, 101, 90),
(2, 2, 102, 85),
(3, 3, 103, 88),
(4, 4, 101, 95);

"""

In [21]:
for result in sql_script.split(';'):

    if result.strip():

        cursor.execute(result)

conn.commit()

print("Tables and Data Inserted Successfully")

Tables and Data Inserted Successfully


In [22]:
query = "SELECT * FROM Students"

cursor.execute(query)

result = cursor.fetchall()

for row in result:
    print(row)

(1, 'Rahul', 'AI', 21, 'Chennai')
(2, 'Kiran', 'Data Science', 22, 'Hyderabad')
(3, 'Anjali', 'Cyber Security', 20, 'Bangalore')
(4, 'Rohit', 'AI', 23, 'Mumbai')


In [23]:
query = """

SELECT
    Students.student_name,
    Courses.course_name,
    Enrollments.marks

FROM Enrollments

INNER JOIN Students
ON Students.student_id = Enrollments.student_id

INNER JOIN Courses
ON Courses.course_id = Enrollments.course_id

"""

cursor.execute(query)

result = cursor.fetchall()

for row in result:
    print(row)

('Rahul', 'Machine Learning', 90)
('Rohit', 'Machine Learning', 95)
('Kiran', 'Data Analytics', 85)
('Anjali', 'Cloud Computing', 88)


In [24]:
query = """

SELECT
    department,
    AVG(marks) AS average_marks

FROM Students

INNER JOIN Enrollments
ON Students.student_id = Enrollments.student_id

GROUP BY department

"""

cursor.execute(query)

result = cursor.fetchall()

for row in result:
    print(row)

('AI', Decimal('92.5000'))
('Data Science', Decimal('85.0000'))
('Cyber Security', Decimal('88.0000'))


In [26]:
cursor.execute("SELECT @@sql_mode")

result = cursor.fetchall()

print(result)

[('ONLY_FULL_GROUP_BY,STRICT_TRANS_TABLES,NO_ZERO_IN_DATE,NO_ZERO_DATE,ERROR_FOR_DIVISION_BY_ZERO,NO_ENGINE_SUBSTITUTION',)]


In [27]:
cursor.execute("""
SET GLOBAL sql_mode=
(
    SELECT REPLACE(
        @@sql_mode,
        'ONLY_FULL_GROUP_BY',
        ''
    )
);
""")

In [28]:
conn.close()

In [29]:
import mysql.connector

conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="root",
    database="student_management"
)

cursor = conn.cursor()

In [30]:
query = """

SELECT
    Students.student_name,
    MAX(Enrollments.marks)

FROM Students

INNER JOIN Enrollments
ON Students.student_id = Enrollments.student_id

"""

cursor.execute(query)

result = cursor.fetchall()

for row in result:
    print(row)

('Rahul', 95)


In [31]:
view_query = """

CREATE VIEW Student_Report AS

SELECT
    Students.student_name,
    Students.department,
    Courses.course_name,
    Enrollments.marks

FROM Students

INNER JOIN Enrollments
ON Students.student_id = Enrollments.student_id

INNER JOIN Courses
ON Courses.course_id = Enrollments.course_id

"""

cursor.execute(view_query)

conn.commit()

print("View Created Successfully")

View Created Successfully


display view

In [32]:
cursor.execute(
    "SELECT * FROM Student_Report"
)

result = cursor.fetchall()

for row in result:
    print(row)

('Rahul', 'AI', 'Machine Learning', 90)
('Rohit', 'AI', 'Machine Learning', 95)
('Kiran', 'Data Science', 'Data Analytics', 85)
('Anjali', 'Cyber Security', 'Cloud Computing', 88)


In [33]:
with open(
    "student_management_mysql.sql",
    "w"
) as file:

    file.write(sql_script)

In [34]:
from google.colab import files

files.download(
    "student_management_mysql.sql"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [35]:
!mysqldump -u root -proot student_management > student_management_dump.sql

mysqldump: [Warning] Using a password on the command line interface can be insecure.


In [36]:
files.download(
    "student_management_dump.sql"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>